# Lab la_01 — Power iteration, SVD low-rank approximation & PCA

**Track 2 · Linear algebra** — skills: *Singular value decomposition*, *PCA & low-rank approximation*.

### Learning goals
1. Find the dominant eigenpair of a symmetric matrix with **power iteration** and see that its convergence rate is $|\lambda_2/\lambda_1|$.
2. Use `np.linalg.svd` to build the **best rank-$k$ approximation** $A_k = \sum_{i\le k}\sigma_i\mathbf u_i\mathbf v_i^\top$
   and verify the **Eckart–Young** error $\|A - A_k\|_F = \sqrt{\sum_{i>k}\sigma_i^2}$.
3. Implement **PCA from scratch** (centre, SVD/eigendecomposition, project, reconstruct) and connect explained variance to singular values.

### Reading
- MIT 18.065 (Strang), *Matrix Methods in Data Analysis, Signal Processing, and Machine Learning*: <https://ocw.mit.edu/courses/18-065-matrix-methods-in-data-analysis-signal-processing-and-machine-learning-spring-2018/>
- MIT 18.06SC, *Singular value decomposition* unit: <https://ocw.mit.edu/courses/18-06sc-linear-algebra-fall-2011/>
- `numpy.linalg.svd`: <https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html>

Fill in every function marked `# TODO`, run the exploration cells, then run the final grading cell. NumPy only.

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## 1. Power iteration

Write $\mathbf v_0 = \sum_i c_i\mathbf q_i$ in the eigenbasis of a symmetric $A$. Then
$$A^k\mathbf v_0 = \lambda_1^k\Big(c_1\mathbf q_1 + \sum_{i\ge 2} c_i(\lambda_i/\lambda_1)^k\mathbf q_i\Big),$$
so after normalising, $\mathbf v_k\to\pm\mathbf q_1$ at rate $|\lambda_2/\lambda_1|$. The **Rayleigh quotient**
$\mathbf v^\top A\mathbf v$ (for unit $\mathbf v$) estimates $\lambda_1$.

Implement: start from a random unit vector (`np.random.default_rng(seed)`), repeat $\mathbf v\leftarrow A\mathbf v/\|A\mathbf v\|$,
stop when the Rayleigh quotient changes by less than `tol` (or after `num_iters`). Return `(eigenvalue, unit_vector)`.

In [ ]:
def power_iteration(A, num_iters=1000, tol=1e-10, seed=0):
    """Top eigenpair (eigenvalue, unit eigenvector) of a symmetric PSD matrix A."""
    # TODO
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(0)
Q, _ = np.linalg.qr(rng.normal(size=(4, 4)))
for gap in [2.0, 1.1, 1.01]:
    A = Q @ np.diag([gap, 1.0, 0.5, 0.1]) @ Q.T
    lam, v = power_iteration(A, num_iters=5000)
    print(f"lambda1/lambda2 = {gap:5}:  lambda ≈ {lam:.6f},  |<v, q1>| = {abs(v @ Q[:, 0]):.6f}")
# Try: how many iterations does each case need? Compare with log(tol)/log(lambda2/lambda1).

## 2. Low-rank approximation of a synthetic "image"

`np.linalg.svd(M, full_matrices=False)` returns `U, s, Vt` with $M = U\,\mathrm{diag}(s)\,V^\top$ and $s$ sorted
in decreasing order. Keep the first $k$ terms. Also write `eckart_young_error(s, k)` that predicts the
Frobenius error **from the singular values alone**.

In [ ]:
def low_rank_approx(M, k):
    """Best rank-k approximation of M via the truncated SVD."""
    # TODO
    raise NotImplementedError


def eckart_young_error(s, k):
    """sqrt(sum_{i>k} s_i^2): Frobenius error of the best rank-k approximation."""
    # TODO
    raise NotImplementedError

In [ ]:
# A 64x64 synthetic image: smooth waves + a rectangle + noise
n = 64
t = np.linspace(0, 1, n)
img = np.outer(np.sin(2 * np.pi * t), np.cos(3 * np.pi * t)) + 0.5 * np.outer(t, t ** 2)
img[20:40, 10:30] += 1.0
img += 0.05 * np.random.default_rng(1).normal(size=img.shape)

s = np.linalg.svd(img, compute_uv=False)
print("first singular values:", s[:8])
for k in [1, 2, 3, 5, 10, 20]:
    Ak = low_rank_approx(img, k)
    print(f"k={k:2d}  error={np.linalg.norm(img - Ak):.4f}  predicted={eckart_young_error(s, k):.4f}  "
          f"storage={k * (2 * n + 1)} vs {n * n} numbers")

In [ ]:
# Optional: look at it (matplotlib is installed; skip this cell if you prefer pure numbers)
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 4, figsize=(12, 3))
    for a, k in zip(ax, [1, 3, 10, None]):
        a.imshow(img if k is None else low_rank_approx(img, k), cmap="gray")
        a.set_title("original" if k is None else f"rank {k}")
        a.axis("off")
    plt.show()
except ImportError:
    pass

## 3. PCA from scratch

For data $X\in\mathbb R^{n\times d}$ (one sample per row): centre $X_c = X - \bar{\mathbf x}$, take the SVD
$X_c = U\Sigma V^\top$. The principal directions are the rows of $V^\top$, and the variances along them are the
eigenvalues of the sample covariance $S = \frac{1}{n-1}X_c^\top X_c$, i.e. $\lambda_i = \sigma_i^2/(n-1)$.

- `pca_fit(X, k)` → `(mean, components, explained_variance)` with `components` of shape `(k, d)` (orthonormal rows, decreasing variance).
- `pca_transform` → coordinates $Z = (X-\bar{\mathbf x})V_k$; `pca_reconstruct` → $\hat X = Z V_k^\top + \bar{\mathbf x}$.
- `explained_variance_ratio(X, k)` → $\sum_{i\le k}\sigma_i^2 / \sum_i\sigma_i^2$.

On paper: why is rank-$k$ PCA reconstruction the same thing as the best rank-$k$ approximation of $X_c$?

In [ ]:
def pca_fit(X, k):
    """Return (mean, components (k, d), explained_variance (k,)) using the 1/(n-1) covariance."""
    # TODO
    raise NotImplementedError


def pca_transform(X, mean, components):
    # TODO
    raise NotImplementedError


def pca_reconstruct(Z, mean, components):
    # TODO
    raise NotImplementedError


def explained_variance_ratio(X, k):
    # TODO
    raise NotImplementedError

In [ ]:
# Synthetic 3-D data that really lives near a tilted 2-D plane
rng = np.random.default_rng(2)
n = 500
latent = rng.normal(size=(n, 2)) * np.array([4.0, 1.5])
R = np.linalg.qr(rng.normal(size=(3, 3)))[0]
X = np.c_[latent, 0.2 * rng.normal(size=n)] @ R.T + np.array([10.0, -3.0, 2.0])

mean, comps, var = pca_fit(X, 3)
print("mean:", mean)
print("explained variance:", var, " (true latent variances ≈ 16, 2.25, 0.04)")
print("ratio k=1,2:", explained_variance_ratio(X, 1), explained_variance_ratio(X, 2))
print("cross-check with np.cov eigenvalues:", np.sort(np.linalg.eigvalsh(np.cov(X, rowvar=False)))[::-1])
m2, c2, _ = pca_fit(X, 2)
Xh = pca_reconstruct(pca_transform(X, m2, c2), m2, c2)
print("rank-2 reconstruction RMS error:", np.sqrt(np.mean((X - Xh) ** 2)))

### Reflection (write on paper)
- Power iteration needs only matrix–vector products. Why is that attractive for a $10^6\times10^6$ sparse matrix?
- Why must PCA centre the data first? What does the first "component" become if you forget?
- If two singular values are equal, what goes wrong with "the" principal direction?

## Grade

In [ ]:
from lc_client import grade
grade("la_01_svd_pca", globals())